In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt

In [ ]:
def run_enrichment_pipeline(deg_csv_path, output_prefix):

    df_genes = pd.read_csv(deg_csv_path, header=None)
    gene_list = df_genes[0].tolist()
    
    
    # KEGG
    enr = gp.enrichr(
            gene_list=gene_list,
            gene_sets='KEGG_2019_Mouse',
            organism='mouse', 
            outdir=None
        )

    results = enr.results
    sig_results = results[results['Adjusted P-value'] < 0.05].copy()
    sig_results = sig_results.sort_values('Adjusted P-value')
    output_csv_path = f"{output_prefix}_KEGG.csv"
    sig_results.to_csv(output_csv_path, index=False)

In [ ]:
run_enrichment_pipeline("ITC_Low_DEGs_List.csv", "ITC_Low")
run_enrichment_pipeline("PTC_Low_DEGs_List.csv", "PTC_Low")
run_enrichment_pipeline("MTC_Low_DEGs_List.csv", "MTC_Low")
run_enrichment_pipeline("PTC_High_DEGs_List.csv", "PTC_High")
run_enrichment_pipeline("ITC_High_DEGs_List.csv", "ITC_High")
run_enrichment_pipeline("MTC_High_DEGs_List.csv", "MTC_High")

In [ ]:
def plot_bidirectional_go(high_csv, low_csv, output_pdf, top_n=5):
    
    df_high = pd.read_csv(high_csv)
    df_low = pd.read_csv(low_csv)

    df_high = df_high.sort_values('Adjusted P-value').head(top_n).copy()
    df_low = df_low.sort_values('Adjusted P-value').head(top_n).copy()

    df_high['Plot_Value'] = -np.log10(df_high['Adjusted P-value'])
    df_high['Color'] = '#e74c3c'  
    df_high['Group'] = 'High'

    df_low['Plot_Value'] = -np.log10(df_low['Adjusted P-value']) * -1
    df_low['Color'] = '#3498db'   
    df_low['Group'] = 'Low'
        
    df_plot = pd.concat([df_low.sort_values('Plot_Value', ascending=True), 
                         df_high.sort_values('Plot_Value', ascending=True)], axis=0).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(5, 0.4 * len(df_plot) + 1.5))   
    bars = ax.barh(
        y=df_plot['Term'], 
        width=df_plot['Plot_Value'], 
        color=df_plot['Color'],
        edgecolor='white',  
        linewidth=1,
        height=0.85         
    )
    
    ax.axvline(0, color='black', linewidth=1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)   
    ax.spines['bottom'].set_linewidth(1.2)
    
    ax.tick_params(axis='x', labelsize=11, width=1.2)
    ax.tick_params(axis='y', labelsize=11, length=0) 

    ax.set_xlabel("Log.q.value", fontsize=13, fontweight='bold', labelpad=8)
    ax.set_title("MTC", fontsize=14, fontweight='bold', pad=15, loc='center') 
    
    plt.tight_layout()
    plt.savefig(output_pdf, format='pdf', bbox_inches='tight', dpi=300)

plot_bidirectional_go("MTC_High_KEGG.csv", "MTC_Low_KEGG.csv", "Fig_MTC_Bidirectional_KEGG_2.pdf",top_n=5)